In [2]:
%load_ext autoreload
%autoreload 2
from kg import PipelineBuilder, EntityMapper
from PyPDF2 import PdfReader
import os
import amrlib
import penman
from concurrent.futures import ThreadPoolExecutor, as_completed
import torch
from amrlib import load_stog_model
import networkx as nx
import sys, os
from pyvis.network import Network
import os
import json
import xml.etree.ElementTree as ET
import psutil
import os
import signal

import epo_ops
import spacy
from dotenv import load_dotenv
from tools.sentence.entity import Entity, InMemoryEntityRepository

from PatentProvider import PatentProvider
import random
import os
import json
from concurrent.futures import ThreadPoolExecutor, as_completed
import json
from collections import Counter
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import json
import json
import random
import os
import spacy
from spacy.tokens import DocBin
import torch
import matplotlib.pyplot as plt
import amrlib
import spacy
from tools.sentence.sentence import Sentence
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from kg.formatting_manager import FormattingManager
import sys, importlib,os
sys.modules.pop('PatentTextFormatter', None)
importlib.invalidate_caches()
import numpy as np
from fastcoref import FCoref
# Import extensions to register spaCy components
from tools.sentence.entity import Entity
import torch, os, platform
from kg.NodeGenerator import NodeGenerator
import spacy
import os, pyvis, jinja2, sys
import epo_ops
import epo_ops
import os
import spacy
import xml.etree.ElementTree as ET
import json
import sys
from PatentProvider import PatentProvider
# Import new graph processing classes
from tools.graph.visualizer import GraphVisualizer
from tools.graph.faiss_merger import FAISSEdgeMerger
from tools.graph.neo4j_manager import Neo4jManager
from kg.formatting_manager import FormattingManager
from tools.sentence.sentence import Sentence
from tools.sentence.sentence_classifier import SentenceClassifier
from dataclasses import dataclass
from typing import List
from kg.ParallelTripleGenerator import ParallelTripleGenerator
import networkx as nx
import logging
import importlib
import graph_validator_chat
#from web_editor.graph_validator_chat import start_validator_chat
from tools.graph.visualizer import GraphVisualizer
# Initialize decomposer
from tools.graph.relation_simplifier import RelationSimplifier
from graph_validator_chat import start_validator_chat
from langfuse import Langfuse



c:\Users\Caleb\Documents\LLM Patent Claim Generator Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
01/26/2026 23:50:15 - INFO - 	 Loading faiss with AVX512 support.
01/26/2026 23:50:15 - INFO - 	 Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
01/26/2026 23:50:15 - INFO - 	 Loading faiss with AVX2 support.
01/26/2026 23:50:15 - INFO - 	 Successfully loaded faiss with AVX2 support.


In [10]:
import torch

if torch.cuda.is_available():
    torch.cuda.set_per_process_memory_fraction(0.7)  # 50% of VRAM

    torch.set_num_threads(4)
    torch.set_num_interop_threads(1)


In [11]:
nlp = spacy.load("en_core_web_trf")   # see optimization ideas below
formatterManager = FormattingManager()
random_description = PatentProvider().getDescription("1502502")

from kg.formatting_manager import FormattingManager

# Initialize the formatting manager
# Default: 8 workers for retrieveContent, 12 for split

# Custom workers
fm = FormattingManager(num_workers=8, split_workers=12)
# Extract only invention-related sentences
sentences = fm.retrieveContent(random_description, chunk_size=1000)

# Returns a list of Sentence objects
for sentence in sentences:
    print(sentence.text)

In order to reduce such labor, a display device is proposed in Japan unexamined patent publication Hei7-230259 provided with pseudo models of aquatic animals such as fish models and the like, and by generating the air bubble from a ventilation member which is installed at the bottom wall of a water tank to generate water flow in a water tank and to make viewers feel as if pseudo models of aquatic animals such as fish models are swimming as a pseudo aquarium in which daily feeding is not required and environmental maintenance such as temperature maintenance and the like are easy
In addition, in peripheral domains where air bubble near the air bubble generating member rises, when the creature models go in said domains, the aquatic animal pseudo models are pushed upward near the water surface by air bubble, they show unnatural behavior, lying down.
The same is applied when pseudo creature models are replaced by models such as submarines other than fish.
The present invention relates to a 

In [12]:

fm = FormattingManager()

# Returns List[Sentence], not List[str]
split_sentences = fm.split(sentences)




In [13]:

# Initialize classifier (uses GPU if available, batch processing enabled)
classifier = SentenceClassifier(
    model_path="training/info/done/hf/sentence_classifier_model",
    batch_size=32,  # Adjust based on GPU memory
    use_gpu=True
)

# Filter sentences to keep only informative ones (much faster with batch processing)
sentence_split = classifier.filter_informative(
    split_sentences, 
    keep_labels=["INFORMATIVE"]  # Can also include ["INFORMATIVE", "FIGURE_RELATED"] if needed
)



SentenceClassifier initialized on device: cuda


In [14]:

@dataclass(frozen=True)
class JoinedText:
    """
    Holds the concatenated text passed to spaCy
    and the starting character offset of each sentence.
    """
    text: str
    starts: List[int]


def join_sentences(sentences, sep=" "):
    """
    Join a list of Sentence objects into one string while
    tracking sentence start offsets.

    Args:
        sentences: List of Sentence objects with `.text`
        sep: Separator inserted between sentences (default: space)

    Returns:
        JoinedText(text, starts)
    """
    parts = []
    starts = []
    cur = 0

    for i, s in enumerate(sentences):
        starts.append(cur)
        parts.append(s.text)
        cur += len(s.text)

        if i < len(sentences) - 1:
            parts.append(sep)
            cur += len(sep)

    return JoinedText("".join(parts), starts)


In [18]:
pipeline_builder = PipelineBuilder()
entity_mapper = EntityMapper(sentence_cls=Sentence)

joined = join_sentences(sentence_split, sep=" ")
doc = pipeline_builder.nlp(joined.text)

clusters = entity_mapper.map_to_sentences(doc, sentence_split, joined)

print("doc.ents:", len(doc.ents))
print("coref clusters:", len(doc._.coref_clusters))
print("entities in first sentence:", len(sentence_split[0].entities))
print(type(sentence_split[0].entities[0]))
print(sentence_split[0].entities[0])

# Collect all entities created by the mapper
all_entities: list[Entity] = []
for sentence in sentence_split:
    all_entities.extend(sentence.entities)

# Create repository and add entities
repo = InMemoryEntityRepository()
for entity in all_entities:
    repo.save(entity)

print("=== REPO CONTENTS ===")
for e in repo.getAll().values():
    print(
        f"Entity("
        f"name={e.name}, "
        f"label={e.label}, "
        f"id={e.id}, "
        f"ref={e.ref}"
        f")"
    )
for ent in doc.ents[:50]:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

# For each sentence, count overlaps
offset = 0
for i, s in enumerate(sentence_split):
    start = offset
    end = start + len(s.text)
    overlaps = [ent for ent in doc.ents if ent.start_char < end and ent.end_char > start]
    print("sentence", i, "overlaps", len(overlaps))
    offset = end + 1



PipelineBuilder: Using GPU (device: cuda)


Device set to use cuda:0


Transformer model detected - GPU acceleration enabled
HF NER initialized on GPU (device=0)
2026-01-25 15:07:16 - fastcoref.modeling - INFO - missing_keys: []
2026-01-25 15:07:16 - fastcoref.modeling - INFO - unexpected_keys: []
2026-01-25 15:07:16 - fastcoref.modeling - INFO - mismatched_keys: []
2026-01-25 15:07:16 - fastcoref.modeling - INFO - error_msgs: []
2026-01-25 15:07:16 - fastcoref.modeling - INFO - Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M
2026-01-25 15:07:25 - fastcoref.modeling - INFO - Tokenize 1 inputs...


Map: 100%|██████████| 1/1 [00:00<00:00, 26.78 examples/s]

2026-01-25 15:07:25 - fastcoref.modeling - INFO - ***** Running Inference on 1 texts *****


...

2026-01-25 15:07:27 - fastcoref.modeling - INFO - Tokenize 1 inputs...



Map: 100%|██████████| 1/1 [00:00<00:00, 110.05 examples/s]

2026-01-25 15:07:27 - fastcoref.modeling - INFO - ***** Running Inference on 1 texts *****



Inference: 100%|██████████| 1/1 [00:00<00:00,  5.03it/s]


doc.ents: 1268
coref clusters: 0
entities in first sentence: 27
<class 'tools.sentence.entity.Entity'>
Entity(display device@4:18)
=== REPO CONTENTS ===
Entity(name=display device, label=HARDWARE, id=None, ref=ent::device::d61489)
Entity(name=fish models, label=COMPONENT, id=None, ref=ent::model::034f09)
Entity(name=aquatic, label=BIOMOLECULE, id=None, ref=ent::aquatic::82bc61)
Entity(name=aquatic animals, label=BIOMOLECULE, id=None, ref=ent::animal::aaf781)
Entity(name=air bubbles, label=COMPONENT, id=None, ref=ent::bubble::3eb283)
Entity(name=air bubble generating member, label=COMPONENT, id=None, ref=ent::member::1affd5)
Entity(name=bottom wall, label=COMPONENT, id=None, ref=ent::wall::82b08f)
Entity(name=water tank, label=COMPONENT, id=None, ref=ent::tank::114045)
Entity(name=liquid flow, label=PROCESS_STEP, id=None, ref=ent::flow::13d26f)
Entity(name=water, label=INVENTION, id=None, ref=ent::water::528a65)
Entity(name=viewers, label=UNCLASSIFIED_ENTITY, id=None, ref=ent::viewer::2

In [ ]:
print(sentence_split)  

In [19]:

# Initialize parallel triple generator
# - 10 workers for parallel processing
# - Rate limit: 900 calls/minute (stays below 1000 limit)
generator = ParallelTripleGenerator(
    repo=repo,max_workers=10,
    rate_limit_per_minute=900,
    verbose=True
)

# Generate triples from sentences (handles all parallelization, rate limiting, etc.)
triples = generator.generate(sentence_split)


Processing 319 sentences in parallel with 10 workers with context...
sentence: The display device is provided with pseudo models of aquatic animals, such as fish models, and the like. [{'id': 'ent::device::d61489', 'label': 'HARDWARE', 'span': [4, 18], 'text': 'display device', 'ref_short': '1489'}, {'id': 'ent::device::d61489', 'label': 'INVENTION', 'span': [4, 18], 'text': 'display device', 'ref_short': '1489'}, {'id': 'ent::member::1affd5', 'label': 'COMPONENT', 'span': [4, 22], 'text': 'ventilation member', 'ref_short': 'ffd5'}, {'id': 'ent::bubble::3eb283', 'label': 'COMPONENT', 'span': [4, 15], 'text': 'air bubbles', 'ref_short': 'b283'}, {'id': 'ent::water::528a65', 'label': 'PROCESS_STEP', 'span': [4, 9], 'text': 'water', 'ref_short': '8a65'}, {'id': 'ent::aquarium::fc39b3', 'label': 'INVENTION', 'span': [4, 19], 'text': 'pseudo aquarium', 'ref_short': '39b3'}, {'id': 'ent::overall::b194f4', 'label': 'INVENTION', 'span': [4, 11], 'text': 'overall', 'ref_short': '94f4'}, {'id': 

In [ ]:
print(triples)

In [20]:
# --- Merge relations per (head, tail) using FAISSEdgeMerger
# Initialize merger
merger = FAISSEdgeMerger(
    sim_threshold=0.8,
    embed_dim=256,
    ngram=3,
    keep="shortest",
)

# Merge relations
triples, merge_stats = merger.merge_relations(triples)

print("Merge stats:", merge_stats)
print("Before:", len(triples), "After:", len(triples))



Merge stats: {'pairs': 163, 'out_triples': 183, 'merged_relations_removed': 0, 'kept_clusters': 183, 'sim_threshold': 0.8, 'embed_dim': 256, 'ngram': 3}
Before: 183 After: 183


In [ ]:


# Use the simplifier (Option 2 - RECOMMENDED)
simplifier = RelationSimplifier(
    max_relation_length=4,
    verbose=True
)

# Simplify triples (keeps same structure, adds properties)
triples = simplifier.simplify(triples)

In [21]:
# --- Build + visualize a typed KG from List[Triple] using GraphVisualizer

# Initialize visualizer
visualizer = GraphVisualizer()

# Build ID -> Name map from sentence_split
id_to_name = visualizer.build_id_to_name_map(sentence_split)

# Build graph from triples
G = visualizer.build_graph(triples)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# Visualize
visualizer.visualize_pyvis(G, out_file="graph_merged.html", id_to_name=id_to_name)


Nodes: 100
Edges: 183
graph_merged.html
✅ Interactive graph written to graph_merged.html


In [32]:
# Save variables
%store triples
%store G
%store sentence_split

Stored 'triples' (list)
Stored 'G' (MultiDiGraph)
Stored 'sentence_split' (list)


In [3]:
# 
# Or reload all stored variables at once
%store -r

In [ ]:

logging.basicConfig(
    level=logging.INFO,  # Use DEBUG for more detail
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

<module 'web_editor.graph_validator_chat.server' from 'c:\\Users\\Caleb\\Documents\\LLM Patent Claim Generator Thesis\\web_editor\\graph_validator_chat\\server.py'>

[API] GET /api/status
[127.0.0.1] "GET /api/status HTTP/1.1" 200 -
[API] GET /api/state
[127.0.0.1] "GET /api/state HTTP/1.1" 200 -
[API] GET /api/triples
[127.0.0.1] "GET /api/triples HTTP/1.1" 200 -


Killing PID 998760


In [6]:
import os
os.system("taskkill /IM node.exe /F")


0

In [10]:
import os
# Kill any existing processes that might be holding port 3000
os.system("taskkill /F /IM node.exe /T")


0

In [ ]:
# Upload current graph to Neo4j (uses .env)
import os
from tools.graph.neo4j_manager import Neo4jManager
from tools.graph.visualizer import GraphVisualizer

try:
    G
except NameError:
    visualizer = GraphVisualizer()
    G = visualizer.build_graph(triples)

try:
    id_to_name
except NameError:
    id_to_name = GraphVisualizer.build_id_to_name_map_from_triples(triples)

neo4j_manager = Neo4jManager(
    uri=os.getenv('NEO4J_URI'),
    user=os.getenv('NEO4J_USERNAME') or os.getenv('NEO4J_USER'),
    password=os.getenv('NEO4J_PASSWORD'),
    database=os.getenv('NEO4J_DATABASE') or None,
)
neo4j_manager.push_multidigraph_to_aura(
    G,
    node_label='Entity',
    rel_type='LINKS_TO',
    id_to_name=id_to_name,
    dedupe_edges=True,
)


Uploaded to Neo4j: nodes=99 edges=181 rel=:LINKS_TO label=:Entity dedupe_edges=True


In [6]:
import os                                                                                                                                                                      
from tools.graph.neo4j_manager import Neo4jManager                                                                                                                             
                                                                                                                                                                                 
print("NEO4J_URI =", os.getenv("NEO4J_URI"))                                                                                                                                   
print("NEO4J_DATABASE =", os.getenv("NEO4J_DATABASE"))                                                                                                                         
                                                                                                                                                                                 
  
mgr = Neo4jManager(                                                                                                                                                            
    uri=os.getenv("NEO4J_URI"),                                                                                                                                                
    user=os.getenv("NEO4J_USERNAME") or os.getenv("NEO4J_USER"),                                                                                                               
    password=os.getenv("NEO4J_PASSWORD"),                                                                                                                                      
    database=os.getenv("NEO4J_DATABASE") or None,                                                                                                                              
  )                                                                                                                                                                              
                                                                                                                                                                                 
print(mgr.run_cypher("MATCH (n) RETURN count(n) AS c")) 

NEO4J_URI = neo4j+s://17ede185.databases.neo4j.io
NEO4J_DATABASE = neo4j
{'records': [{'c': 99}], 'keys': ['c'], 'summary': {'nodes_created': 0, 'nodes_deleted': 0, 'relationships_created': 0, 'relationships_deleted': 0, 'properties_set': 0, 'labels_added': 0, 'labels_removed': 0}}


In [ ]:
# Start Next.js + API without preloading prior graph data
import importlib
import graph_validator_chat.server as gvs

# Stop any running servers
gvs.stop_validator_chat()

# Clear in-memory validator state to avoid reusing old graph
gvs.validator = None
gvs._sentence_split = None

# Reload server module to reset globals
gvs = importlib.reload(gvs)
from graph_validator_chat import start_validator_chat

APP_PORT = 50025
start_validator_chat(port=APP_PORT, debug=True)


In [5]:
from IPython.display import Javascript, display
import os
# Kill any existing node processes that might be holding port 3000
os.system("taskkill /F /IM node.exe /T")
display(Javascript("""
Jupyter.notebook.execute_cells([0])
"""))

import importlib
importlib.reload(graph_validator_chat.server)
import os
from pyngrok import ngrok, conf
import psutil  # Added: Import psutil
import signal # Added: Import signal

# Define the application port consistently
APP_PORT = 50025 # This should match the port in start_validator_chat
 
killed_pids = set()

# Update the PID killing logic to use APP_PORT
for conn in psutil.net_connections(kind="inet"):
    if conn.laddr and conn.laddr.port == APP_PORT and conn.status == psutil.CONN_LISTEN:
        pid = conn.pid
        if pid and pid not in killed_pids:
            print(f"Killing PID {pid}")
            os.kill(pid, signal.SIGTERM)
            killed_pids.add(pid)
        
%store -r
# Fix Jinja2 compatibility - run this FIRST
import jinja2
if not hasattr(jinja2, ''):
    from markupsafe import escape
    jinja2.escape = escape
    import logging
import sys
import logging
# Configure logging to output to stdout (visible in Jupyter cells)
logging.basicConfig(
    level=logging.INFO,  # or logging.DEBUG for more detail
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    stream=sys.stdout,  # This ensures output goes to the cell
    force=True  # Override any existing c
)

# Now your logging will appear in the cell
# OP
# Now your normal imports will work
from tools.graph.kg_gen_converter import build_id_to_name_map
# Build id_to_name mapping
id_to_name = build_id_to_name_map(triples) # Assuming 'triples' is defined elsewhere

# --- NGROK SETUP (MOVED BEFORE start_validator_chat) ---
# Set ngrok authtoken BEFORE connecting
ngrok.set_auth_token(os.getenv("NGROK_AUTH_TOKEN"))

cfg = conf.get_default()
cfg.region = "eu"          # <--- key change

# Start ngrok tunnel, connecting to APP_PORT
try:
    #public_url = ngrok.connect(3000) # Connect ngrok to the same APP_PORT
    #print(f"Ngrok tunnel established at: {public_url}")
    print(f"You can access your local server via this URL.")
except Exception as e:
    print(f"Error starting ngrok tunnel: {e}")
    # You might want to add error handling here, e.g., if ngrok is not found
# --- END NGROK SETUP ---

# Start chat (browser opens automatically)
# Use LangGraph validator (default)

# Enable debug mode
start_validator_chat(
    graph=G, # Assuming 'G' is defined elsewhere
    sentence_split=sentence_split, # Assuming 'sentence_split' is defined elsewhere
    triples=triples,
    id_to_name=id_to_name,
    debug=True,
    port = APP_PORT  # Ensure your application uses the same APP_PORT
)

<IPython.core.display.Javascript object>

2026-01-26 23:51:44 - pyngrok.process - INFO - Updating authtoken for default "config_path" of "ngrok_path": C:\Users\Caleb\AppData\Local\ngrok\ngrok.exe
You can access your local server via this URL.
[Server] Requested API port: 50025
✓ Using requested port 50025 for API server
✓ API Server running on http://localhost:50025
✓ Using port 3000 for Next.js frontend
🚀 Starting Next.js dev server on port 3000...
✓ Graph Validator Chat: http://localhost:3000
✓ API Server: http://localhost:50025
[Server] Initializing validator in background...

[Server] ✓ Servers are running. This cell will stay ACTIVE.
[Server] ✓ Debugger will remain attached while this loop runs.
[Server] ✓ Press Ctrl+C in this cell to stop servers.
[Server] Entering blocking loop (checking every 0.5s)...
[Server] Current time: 23:51:45
[Server] About to enter loop.
[Server] _server_running = True
[Server] api thread alive = True
[Server] nextjs thread alive = True
[Server] Starting background analysis...
[Server] Opening 

In [ ]:
# Reload the module to get the latest changes
import importlib
import sys

# Remove the module from cache
# Note: claim_drafting_agent, claim_extractor, claim_concept_agent, assertion_agent, and cluster_manager have been removed
if 'tools.graph.rag.graph_rag' in sys.modules:
    del sys.modules['tools.graph.rag.graph_rag']
if 'tools.graph' in sys.modules:
    del sys.modules['tools.graph']

# Re-import (only GraphRAG remains - other classes have been removed)
from tools.graph import GraphRAG

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch 

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained("lj408/PatClaimEval-Quality", trust_remote_code=True)
model = AutoModel.from_pretrained("lj408/PatClaimEval-Quality", trust_remote_code=True).to(device)
gold_claim = "1. A computer-implemented method comprising: identifying a primary code segment; ..."
candidate_claim = "1. A computer-implemented method for managing logger source code segments in a source code development platform, ..."
res = model.score_pair(gold_claim, candidate_claim , tokenizer, device)
print(res)